# 10장 직선을 넘어서: 다층 신경망 — 뉴런, XOR, 활성화 함수, MLPClassifier
『머신러닝과 딥러닝 기초』 실습 노트북

다음을 해 봅니다.
1. 계단 함수 뉴런 하나로 AND, OR, NAND 만들기
2. 뉴런 세 개(2-2-1 신경망)로 XOR 풀기
3. 시그모이드와 ReLU로 XOR 다시 풀기, 활성화 함수 그려 보기
4. 반달 데이터를 `MLPClassifier`로 분류하고 은닉 뉴런 수 바꾸기
5. 손글씨 숫자를 64-64-10 신경망으로 분류하기

위에서부터 차례로 실행하세요.

## 1. 뉴런 하나로 논리 연산
뉴런은 가중합 $z = w_1x_1 + w_2x_2 + b$를 활성화 함수에 넣습니다. 계단 함수는 $z > 0$이면 1, 아니면 0입니다. 출력의 네 숫자는 입력 (0,0), (0,1), (1,0), (1,1)에 대한 답입니다.

In [ ]:
import numpy as np

def step(z):                      # 계단 함수: z가 0보다 크면 1, 아니면 0
    return (z > 0).astype(int)

def neuron(x1, x2, w1, w2, b, act=step):
    return act(w1 * x1 + w2 * x2 + b)    # 가중합에 활성화 함수를 씌운다

x1 = np.array([0, 0, 1, 1])       # 네 가지 입력 (0,0), (0,1), (1,0), (1,1)
x2 = np.array([0, 1, 0, 1])
print("AND :", neuron(x1, x2, 1, 1, -1.5))
print("OR  :", neuron(x1, x2, 1, 1, -0.5))
print("NAND:", neuron(x1, x2, -1, -1, 1.5))

## 2. 층을 쌓아 XOR 풀기
XOR은 OR와 NAND의 AND입니다. 은닉 뉴런 $h_1$(OR), $h_2$(NAND)의 출력을 출력 뉴런(AND)에 넣습니다.

In [ ]:
h1 = neuron(x1, x2, 1, 1, -0.5)       # 은닉 뉴런 1: OR
h2 = neuron(x1, x2, -1, -1, 1.5)      # 은닉 뉴런 2: NAND
y_hat = neuron(h1, h2, 1, 1, -1.5)    # 출력 뉴런: h1과 h2의 AND
print("h1 :", h1)
print("h2 :", h2)
print("XOR:", y_hat)

## 3. 활성화 함수
계단 함수는 기울기가 0이라 경사 하강법으로 학습할 수 없습니다. 가중치와 편향을 10배로 키워 시그모이드로 바꾸면 거의 같은 답을 냅니다.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

h1 = neuron(x1, x2, 10, 10, -5, act=sigmoid)    # 가중치와 편향을 10배로
h2 = neuron(x1, x2, -10, -10, 15, act=sigmoid)
y_hat = neuron(h1, h2, 10, 10, -15, act=sigmoid)
print("h1 :", h1.round(3))
print("h2 :", h2.round(3))
print("XOR:", y_hat.round(3))

ReLU 은닉 뉴런 두 개와 활성화 함수 없는 출력 $h_1 - 2h_2$로도 XOR이 풀립니다(꺾은선).

In [ ]:
def relu(z):
    return np.maximum(0, z)

h1 = neuron(x1, x2, 1, 1, 0, act=relu)     # ReLU(x1 + x2)
h2 = neuron(x1, x2, 1, 1, -1, act=relu)    # ReLU(x1 + x2 - 1)
print("h1 :", h1)
print("h2 :", h2)
print("XOR:", h1 - 2 * h2)                 # 출력은 활성화 함수 없이 가중합만

시그모이드, tanh, ReLU와 그 도함수를 그려 봅니다(이북 그림 10.4). 시그모이드 기울기의 최댓값은 0.25, tanh는 1, ReLU는 양수에서 늘 1입니다.

In [ ]:
import matplotlib.pyplot as plt

zz = np.linspace(-4, 4, 400)
s = sigmoid(zz)
fig, axs = plt.subplots(2, 3, figsize=(10, 5), sharex=True)
for j, (name, f, d) in enumerate([("sigmoid", s, s * (1 - s)),
                                  ("tanh", np.tanh(zz), 1 - np.tanh(zz) ** 2),
                                  ("ReLU", relu(zz), (zz > 0).astype(float))]):
    axs[0, j].plot(zz, f); axs[0, j].set_title(name)
    axs[1, j].plot(zz, d, ls="--"); axs[1, j].set_ylim(-0.05, 1.1)
    axs[1, j].set_xlabel("z")
axs[0, 0].set_ylabel("function"); axs[1, 0].set_ylabel("derivative")
plt.tight_layout(); plt.show()

## 4. 반달 데이터
9장과 같은 반달 데이터(200개)와 같은 나눔입니다. 9장에서 로지스틱 회귀는 테스트 정확도 90%, RBF 커널 SVM은 94%였습니다. 은닉 뉴런 수를 바꾸어 봅니다.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neural_network import MLPClassifier

Xm, ym = make_moons(n_samples=200, noise=0.2, random_state=0)  # 9장의 반달
Ma, Mb, ma, mb = train_test_split(
    Xm, ym, test_size=0.25, random_state=0, stratify=ym)
for h in [1, 2, 4, 10, 100]:
    mlp = make_pipeline(StandardScaler(),
                        MLPClassifier(hidden_layer_sizes=(h,), solver="lbfgs",
                                      max_iter=1000, random_state=0))
    mlp.fit(Ma, ma)
    print(f"은닉 뉴런 {h:3d}개: 학습 {mlp.score(Ma, ma):.1%}, "
          f"테스트 {mlp.score(Mb, mb):.1%}")

결정 경계를 그려 봅니다(이북 그림 10.6). 뉴런이 늘수록 경계가 여러 번 꺾입니다.

In [ ]:
def plot_boundary(model, ax, title=""):
    gx, gy = np.meshgrid(np.linspace(-1.6, 2.6, 300), np.linspace(-1.2, 1.7, 300))
    G = np.c_[gx.ravel(), gy.ravel()]
    P = model.predict_proba(G)[:, 1].reshape(gx.shape)
    ax.contourf(gx, gy, P, levels=[0, 0.5, 1], colors=["tab:blue", "tab:red"], alpha=0.15)
    ax.contour(gx, gy, P, levels=[0.5], colors="k")
    ax.scatter(Ma[:, 0], Ma[:, 1], c=ma, cmap="bwr", s=8)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])

fig, axs = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, h in zip(axs, [1, 2, 4, 10]):
    m = make_pipeline(StandardScaler(),
                      MLPClassifier(hidden_layer_sizes=(h,), solver="lbfgs",
                                    max_iter=1000, random_state=0)).fit(Ma, ma)
    plot_boundary(m, ax, f"hidden {h}: test {m.score(Mb, mb):.0%}")
plt.show()

학습한 신경망의 가중치 모양과 파라미터 수, 활성화 함수를 봅니다. 사이킷런은 시그모이드를 `logistic`이라고 부릅니다.

In [ ]:
mlp = make_pipeline(StandardScaler(),
                    MLPClassifier(hidden_layer_sizes=(10,), solver="lbfgs",
                                  max_iter=1000, random_state=0)).fit(Ma, ma)
net = mlp[-1]                                   # 파이프라인의 마지막 단계
print("가중치 모양:", [W.shape for W in net.coefs_])
print("편향 개수  :", [len(c) for c in net.intercepts_])
n = sum(W.size for W in net.coefs_) + sum(len(c) for c in net.intercepts_)
print("파라미터 수:", n)
print("활성화 함수: 은닉층", net.activation, "/ 출력층", net.out_activation_)

처음 가중치를 정하는 난수(`random_state`)를 바꾸면 결과가 조금 달라집니다.

In [ ]:
for rs in range(5):
    m = make_pipeline(StandardScaler(),
                      MLPClassifier(hidden_layer_sizes=(10,), solver="lbfgs",
                                    max_iter=1000, random_state=rs)).fit(Ma, ma)
    print(f"random_state={rs}: 학습 {m.score(Ma, ma):.1%}, 테스트 {m.score(Mb, mb):.1%}")

## 5. 손글씨 숫자
6장과 같은 손글씨 숫자 데이터와 나눔입니다. 64-64-10 신경망의 출력층은 소프트맥스입니다. 같은 나눔에서 소프트맥스 회귀와도 비교합니다.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression

Xd, yd = load_digits(return_X_y=True)          # 1797장 × 64칸 (6장)
Dtr, Dte, dtr, dte = train_test_split(
    Xd, yd, test_size=0.25, random_state=42, stratify=yd)  # 6장과 같은 나눔
dnet = make_pipeline(StandardScaler(),
                     MLPClassifier(hidden_layer_sizes=(64,), random_state=0))
dnet.fit(Dtr, dtr)
print(f"신경망: 학습 {dnet.score(Dtr, dtr):.1%}, 테스트 {dnet.score(Dte, dte):.1%}")
print("가중치 모양:", [W.shape for W in dnet[-1].coefs_])
print("출력층 활성화 함수:", dnet[-1].out_activation_)
print("첫 이미지의 확률:")
print(dnet.predict_proba(Dte[:1]).round(3))
print("정답:", dte[0])
soft = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
soft.fit(Dtr, dtr)
print(f"소프트맥스 회귀: 테스트 {soft.score(Dte, dte):.1%}")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 3))
axs[0].imshow(Dte[0].reshape(8, 8), cmap="gray_r"); axs[0].set_title(f"label {dte[0]}")
axs[1].bar(range(10), dnet.predict_proba(Dte[:1])[0]); axs[1].set_xticks(range(10))
axs[1].set_xlabel("digit"); axs[1].set_ylabel("probability")
plt.show()

## 6. TensorFlow Playground
https://playground.tensorflow.org 에서 XOR 모양 데이터를 고르고 은닉층을 0개 → 1개(뉴런 2개, 4개)로 바꾸어 학습해 보세요. 활성화 함수를 Linear로 바꾸면 어떻게 되는지도 확인하세요(이북 10.6절).

---
## 실습 문제

**문제 1.** 반달 데이터에서 은닉 뉴런 10개 신경망의 은닉층 활성화 함수(`activation`)를 `"identity"`(활성화 함수 없음), `"logistic"`, `"tanh"`, `"relu"`로 바꾸어 학습·테스트 정확도를 비교하세요. `"identity"`의 결과는 9장의 로지스틱 회귀(테스트 90%)와 어떤가요? 그 까닭은 무엇인가요?

In [ ]:
# 문제 1 풀이

**문제 2.** 반달 데이터에서 `hidden_layer_sizes`를 `(4,)`, `(4, 4)`, `(10,)`, `(10, 10)`, `(100,)`으로 바꾸어 파라미터 수와 학습·테스트 정확도를 출력하세요. 파라미터 수는 `coefs_`와 `intercepts_`로 세고, 손으로 계산한 값과 맞는지 확인하세요.

In [ ]:
# 문제 2 풀이

**문제 3.** 활성화 함수가 `"tanh"`이고 은닉 뉴런이 2개인 `MLPClassifier`(`solver="lbfgs"`)로 XOR의 네 점을 학습하세요. `random_state`를 0\~9로 바꾸며 네 점을 모두 맞히는 경우가 몇 번인지 세어 보세요. 실패한 경우는 왜 생길까요?

In [ ]:
# 문제 3 풀이

**문제 4.** 손글씨 숫자에서 은닉 뉴런을 16, 32, 64, 128개로 바꾸어 파라미터 수와 테스트 정확도를 비교하세요. 그리고 64개짜리 신경망(`dnet`)이 틀린 테스트 이미지 몇 장을 정답·예측과 함께 그려 보세요.

In [ ]:
# 문제 4 풀이